In [ ]:
import pandas as pd
import numpy as np
import warnings

# Ignorar avisos inofensivos para manter o teu relatório limpo
warnings.filterwarnings('ignore')

def pipeline_higienizacao_precos(caminho_arquivo, caminho_saida, limite_tolerancia_nans=0.15):
    """
    Higieniza a matriz de preços, filtra ativos ilíquidos, remove anomalias via IQR e imputa dados.
    
    Parâmetros:
    - caminho_arquivo: Local do teu CSV bruto (ex: precos_sanitizada_v3.csv)
    - caminho_saida: Onde queres guardar o CSV limpo.
    - limite_tolerancia_nans: Percentagem máxima de dias sem preço tolerada (15% por defeito).
    """
    print("=== INÍCIO DO PIPELINE METODOLÓGICO DE PREÇOS ===")
    
    # ---------------------------------------------------------
    # FASE 1: GOVERNANÇA E ESTRUTURAÇÃO
    # ---------------------------------------------------------
    print("1. A carregar e a estruturar a base matricial...")
    # Lemos o CSV. O index_col=0 já define a 'Data' como índice se ela for a primeira coluna
    df = pd.read_csv(caminho_arquivo)
    
    # Garantir que a coluna de data se chama 'Data' e convertê-la para o formato temporal oficial
    if 'Data' in df.columns:
        df['Data'] = pd.to_datetime(df['Data'], errors='coerce')
        df = df.dropna(subset=['Data']) # Remove linhas onde a data não foi lida corretamente
        df.set_index('Data', inplace=True)
    
    # Ordenar cronologicamente
    df.sort_index(inplace=True)
    
    total_linhas_iniciais = len(df)
    total_ativos_iniciais = len(df.columns)
    
    # ---------------------------------------------------------
    # FASE 2: FILTRO ESTATÍSTICO DE OMISSÕES (LIQUIDEZ)
    # ---------------------------------------------------------
    print(f"2. A aplicar crivo de tolerância de dados em falta (Max: {limite_tolerancia_nans*100}%)...")
    
    # Calcula a percentagem de NaNs para cada ativo
    percentagem_nans = df.isna().mean()
    
    # Identifica os ativos que passaram no teste
    ativos_aprovados = percentagem_nans[percentagem_nans <= limite_tolerancia_nans].index
    ativos_reprovados = percentagem_nans[percentagem_nans > limite_tolerancia_nans].index
    
    # Filtra o DataFrame apenas com os aprovados
    df = df[ativos_aprovados]
    
    print(f"   -> Ativos aprovados: {len(ativos_aprovados)}")
    print(f"   -> Ativos reprovados (excluídos): {len(ativos_reprovados)}")
    
    # ---------------------------------------------------------
    # FASE 3: ESCRUTÍNIO DE ANOMALIAS (OUTLIERS VIA IQR)
    # ---------------------------------------------------------
    print("3. A processar o rastreio Interquartil (IQR) para remoção de Outliers extremos...")
    outliers_totais = 0
    
    for col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Como estamos a lidar com preços de ações (que podem crescer muito no longo prazo),
        # um multiplicador de 1.5 pode ser muito agressivo. Vamos usar 3.0 (Outliers Severos).
        limite_inferior = Q1 - 3.0 * IQR
        limite_superior = Q3 + 3.0 * IQR
        
        # Onde o preço ultrapassa o limite OU é menor que zero (preço negativo é impossível)
        mascara_anomalias = (df[col] < limite_inferior) | (df[col] > limite_superior) | (df[col] < 0)
        outliers_totais += mascara_anomalias.sum()
        
        # Suprimir a anomalia (transformar em NaN)
        df.loc[mascara_anomalias, col] = np.nan

    print(f"   -> Anomalias suprimidas cirurgicamente: {outliers_totais} ocorrências.")

    # ---------------------------------------------------------
    # FASE 4: IMPUTAÇÃO HEURÍSTICA CONTROLADA
    # ---------------------------------------------------------
    print("4. A executar imputação temporal matemática...")
    
    # Interpolação linear temporal (cria uma ponte entre pontos válidos ao longo do tempo)
    df = df.interpolate(method='time')
    
    # Preenchimento das pontas (arrasta valores para a frente e para trás para fechar lacunas nas bordas)
    df = df.ffill()
    
    nans_finais = df.isna().sum().sum()
    print(f"   -> Lacunas preenchidas. Valores nulos remanescentes: {nans_finais}")

    # ---------------------------------------------------------
    # FASE 5: EXPORTAÇÃO
    # ---------------------------------------------------------
    print("5. A gerar arquivo final...")
    df.to_csv(caminho_saida)
    
    print("\n=== RESUMO METODOLÓGICO DA AMOSTRA ===")
    print(f"Pregões Analisados: {total_linhas_iniciais}")
    print(f"Universo Inicial de Ativos: {total_ativos_iniciais}")
    print(f"Amostra Final de Ativos Retidos: {len(df.columns)}")
    print(f"Arquivo exportado para: {caminho_saida}")
    print("===================================================")

    return df

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
# Substitui com o caminho onde tens o teu ficheiro precos_sanitizada_v3.csv
caminho_entrada = r"C:\VSCodeWorkspace\TCC_Escrito\data\precos_sanitizada_v3.csv"
caminho_saida = r"C:\VSCodeWorkspace\TCC_Escrito\data\precos_limpos_finais.csv"

# Executar a função! Podes mudar o 0.15 para 0.10 se quiseres ser mais rigoroso (máx 10% de falhas)
base_precos_limpa = pipeline_higienizacao_precos(caminho_entrada, caminho_saida, limite_tolerancia_nans=0.05)

=== INÍCIO DO PIPELINE METODOLÓGICO DE PREÇOS ===
1. A carregar e a estruturar a base matricial...
2. A aplicar crivo de tolerância de dados em falta (Max: 5.0%)...
   -> Ativos aprovados: 136
   -> Ativos reprovados (excluídos): 0
3. A processar o rastreio Interquartil (IQR) para remoção de Outliers extremos...
   -> Anomalias suprimidas cirurgicamente: 7443 ocorrências.
4. A executar imputação temporal matemática...
   -> Lacunas preenchidas. Valores nulos remanescentes: 0
5. A gerar arquivo final...

=== RESUMO METODOLÓGICO DA AMOSTRA ===
Pregões Analisados: 4030
Universo Inicial de Ativos: 136
Amostra Final de Ativos Retidos: 136
Arquivo exportado para: C:\VSCodeWorkspace\TCC_Escrito\data\precos_limpos_finais.csv
